# Phase 2 사전 준비: 자치구(gu)별 데이터 분할

이 노트북은 전 단계인 Phase 1에서 필터링된 전체 데이터(`02_keyword_filtered.csv`)를 읽어와서, 각 자치구(`gu`) 단위로 데이터를 쪼개는 작업을 수행합니다.

- 구별 데이터 개수를 확인하여, 이후 LLM API(Track A)와 SLM 파인튜닝(Track B) 중 어떤 방식을 선택할지 결정하는 기준이 됩니다.
- 분할된 데이터는 `data/processed/split_by_gu/` 폴더 내에 `{구이름}.csv` 형식으로 저장됩니다.

In [ ]:
import os
import pandas as pd
from pathlib import Path

# 파일 경로 설정
INPUT_FILE = "../data/processed/02_keyword_filtered.csv"
INPUT_FILE_NO_FILTER="../data/processed/01_cleaned_merged.csv"
OUTPUT_DIR = "../data/processed/split_by_gu"

print(f"입력 파일: {INPUT_FILE}")
print(f"출력 폴더: {OUTPUT_DIR}")

## 1. 데이터 로드 및 분포 확인

In [ ]:
# 1. 필터링된 데이터 로드 (02_keyword_filtered.csv)
df_filtered = pd.read_csv(INPUT_FILE)
df_filtered['gu'] = df_filtered['gu'].fillna('알수없음')
count_filtered = df_filtered['gu'].value_counts()

# 2. 원본(정제된) 데이터 로드 (01_cleaned_merged.csv)
df_raw = pd.read_csv(INPUT_FILE_NO_FILTER)
df_raw['gu'] = df_raw['gu'].fillna('알수없음')
count_raw = df_raw['gu'].value_counts()

total_filtered = len(df_filtered)
total_raw = len(df_raw)

print(f"전체 원본 데이터: {total_raw:,}건")
print(f"키워드 필터링 데이터: {total_filtered:,}건 ({(total_filtered/total_raw)*100:.1f}% 남음)\n")

print("=== 자치구(gu)별 데이터 분포 비교 ===")
print(f"{'자치구':<8} | {'원본 데이터':>10} | {'필터링 후 데이터':>12} | {'생존율(%)'}")
print("-" * 55)

# 필터링 후 데이터 수 기준으로 내림차순 정렬하여 출력
for gu, f_count in count_filtered.items():
    r_count = count_raw.get(gu, 0)
    survival_rate = (f_count / r_count) * 100 if r_count > 0 else 0
    print(f"{gu:<10}| {r_count:>10,}건 | {f_count:>12,}건 | {survival_rate:5.1f}%")

# 아래 분할 저장을 위해 df 변수명을 df_filtered로 매핑해줌
df = df_filtered


## 2. 자치구별 파일 생성 및 저장

In [ ]:
# 출력 디렉토리 생성
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("구별 파일 저장을 시작합니다...\n")

# 구(gu)를 기준으로 그룹화하여 각각의 파일로 저장
for gu, group_df in df.groupby('gu'):
    # 파일명에 문제가 생기지 않도록 영숫자/한글 및 띄어쓰기만 남김
    safe_gu = "".join(c for c in str(gu) if c.isalnum() or c in (' ', '-', '_')).strip()
    output_file = os.path.join(OUTPUT_DIR, f"{safe_gu}.csv")
    
    # 파일 저장
    group_df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"[저장 완료] {safe_gu}.csv (데이터 크기: {len(group_df):,}건)")

print("\n모든 자치구 파일이 성공적으로 분할 되었습니다!")